# 1: training/test dataset, learning rate, normalization

## normalize 하지 않은 feature

In [ ]:
import tensorflow as tf
import numpy as np

# dtype='float32'을 지정해줘야 내적에서 오류가 안남
xy = np.array([[828.659973, 833.450012, 908100, 828.349976, 831.659973],
               [823.02002, 828.070007, 1828100, 821.655029, 828.070007],
               [819.929993, 824.400024, 1438100, 818.97998, 824.159973],
               [816, 820.958984, 1008100, 815.48999, 819.23999],
               [819.359985, 823, 1188100, 818.469971, 818.97998],
               [819, 823, 1198100, 816, 820.450012],
               [811.700012, 815.25, 1098100, 809.780029, 813.669983],
               [809.51001, 816.659973, 1398100, 804.539978, 809.559998]], dtype='float32')

X = xy[:, 0:-1]
Y = xy[:, [-1]]

In [ ]:
nx = X.shape[1]

W = tf.Variable(tf.random.normal([nx, 1]))
b = tf.Variable(tf.random.normal([1]))

epochs = 1
learning_rate = 0.00001
for step in range(epochs):
    with tf.GradientTape() as tape:
        hypothesis = tf.matmul(X, W) + b
        cost = tf.reduce_mean(tf.square(hypothesis - Y))

    W_grad, b_grad = tape.gradient(cost, [W, b])

    W.assign_sub(learning_rate * W_grad)
    b.assign_sub(learning_rate * b_grad)
    if step % 200 == 0:
        print(f"{step:>4} {cost.numpy():>5.5f}")

tf.Tensor(
[[1.3545863e+07]
 [1.3625071e+07]
 [2.1997289e+10]
 [1.3515360e+07]], shape=(4, 1), dtype=float32) tf.Tensor([16552.713], shape=(1,), dtype=float32)
   0 73110080.00000


feature의 스케일이 너무 차이가 크다면 cost가 nan으로 측정 불가가 된다.

## Normalized inputs (min-max scale)

In [ ]:
import tensorflow as tf
import numpy as np

xy = np.array([[828.659973, 833.450012, 908100, 828.349976, 831.659973],
               [823.02002, 828.070007, 1828100, 821.655029, 828.070007],
               [819.929993, 824.400024, 1438100, 818.97998, 824.159973],
               [816, 820.958984, 1008100, 815.48999, 819.23999],
               [819.359985, 823, 1188100, 818.469971, 818.97998],
               [819, 823, 1198100, 816, 820.450012],
               [811.700012, 815.25, 1098100, 809.780029, 813.669983],
               [809.51001, 816.659973, 1398100, 804.539978, 809.559998]], dtype='float32')

In [ ]:
# normalize 함수 정의
def min_max_scaler(data):
    numerator = data - np.min(data, 0)
    denominator = np.max(data, 0) - np.min(data, 0)
    # noise term prevents the zero division
    return numerator / (denominator + 1e-7)

scaled_xy = min_max_scaler(xy)
scaled_xy

array([[1.        , 1.        , 0.        , 1.        , 1.        ],
       [0.70548487, 0.70439553, 1.        , 0.71881783, 0.8375579 ],
       [0.5441255 , 0.50274825, 0.57608694, 0.606468  , 0.6606331 ],
       [0.33890355, 0.31368026, 0.10869565, 0.45989135, 0.4380092 ],
       [0.51436   , 0.4258239 , 0.3043478 , 0.5850481 , 0.42624405],
       [0.4955618 , 0.4258239 , 0.3152174 , 0.48131135, 0.49276137],
       [0.11436066, 0.        , 0.20652173, 0.22007777, 0.1859724 ],
       [0.        , 0.077471  , 0.5326087 , 0.        , 0.        ]],
      dtype=float32)

In [ ]:
# sklearn 내장 함수 사용
from sklearn.preprocessing import MinMaxScaler
scaled_xy = MinMaxScaler().fit_transform(xy)
scaled_xy

array([[1.        , 1.        , 0.        , 1.        , 1.        ],
       [0.7054825 , 0.7043953 , 1.        , 0.71881485, 0.83755875],
       [0.5441246 , 0.5027466 , 0.57608694, 0.6064682 , 0.6606331 ],
       [0.33890152, 0.31367874, 0.10869569, 0.45988846, 0.43801117],
       [0.5143585 , 0.4258232 , 0.3043478 , 0.5850487 , 0.42624283],
       [0.4955597 , 0.4258232 , 0.31521744, 0.4813118 , 0.49276352],
       [0.11436081, 0.        , 0.20652169, 0.22007751, 0.18597412],
       [0.        , 0.07747269, 0.5326087 , 0.        , 0.        ]],
      dtype=float32)

In [ ]:
X = xy[:, 0:-1]
Y = xy[:, [-1]]

nx = X.shape[1]

W = tf.Variable(tf.random.normal([nx, 1]))
b = tf.Variable(tf.random.normal([1]))

epochs = 1001
learning_rate = 0.01
for step in range(epochs):
    with tf.GradientTape() as tape:
        hypothesis = tf.matmul(X, W) + b
        cost = tf.reduce_mean(tf.square(hypothesis - Y))

    W_grad, b_grad = tape.gradient(cost, [W, b])
    
    W.assign_sub(learning_rate * W_grad)
    b.assign_sub(learning_rate * b_grad)
    if step % 200 == 0:
        print(f"{step:>4} {cost.numpy():>5.5f}")

   0 2.61133
 200 0.29227
 400 0.15499
 600 0.09027
 800 0.05712
1000 0.03927


이제 cost가 nan이 나오지 않고 0으로 잘 수렴하는 것을 볼 수 있다. 

# 2. Meet MNIST Dataset


손글씨 이미지 하나의 shape은 28x28로 총 784개의 feature라고 볼 수 있다.  
레이블은 0부터 9까지 총 10개이다.

## 1. 데이터 로드 & 전처리

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

In [ ]:
x_train = x_train / 255,0
x_test = x_test / 255.0

## 2. 알고리즘 선택

10가지 손글씨를 분류하는 문제이기 때문에 classifier를 사용한다. 

레이블이 10개로 다중 분류이니까 softmax를 사용한다.

logistic regression (softmax)를 사용한다.

In [ ]:
# 모델 선언
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)), # 인풋이 (28, 28)이니까 이걸 784로 쭉 펼쳐준다. 
    keras.layers.Dense(128, activation=tf.nn.relu),
    keras.layers.Dense(10, activation=tf.nn.softmax)
])

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 128)               100480    
                                                                 
 dense_1 (Dense)             (None, 10)                1290      
                                                                 
Total params: 101,770
Trainable params: 101,770
Non-trainable params: 0
_________________________________________________________________


In [ ]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 [==============================] - 6s 3ms/step - loss: 2.3214 - accuracy: 0.8536
Epoch 2/5
1875/1875 [==============================] - 5s 3ms/step - loss: 0.3790 - accuracy: 0.9064
Epoch 3/5
1875/1875 [==============================] - 5s 3ms/step - loss: 0.2854 - accuracy: 0.9286
Epoch 4/5
1875/1875 [==============================] - 5s 3ms/step - loss: 0.2409 - accuracy: 0.9380
Epoch 5/5
1875/1875 [==============================] - 5s 3ms/step - loss: 0.2247 - accuracy: 0.9424


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
predictions = model.predict(x_test)
np.argmax(predictions[0])

313/313 [==============================] - 1s 2ms/step - loss: 0.2661 - accuracy: 0.9446


7